In [1]:
import contextlib, math, time
from tinygrad import Tensor, nn, dtypes
from tinygrad.helpers import GlobalCounters, colored

In [2]:
def time_to_str(t:float, w=18) -> str: return colored(next((f"{t * d:{w-len(pr)-1}.2f} {pr}" for d,pr in [(1, "s "),(1e3, "ms")] if t > 10/d), f"{t * 1e6:{w-4}.2f} us"), 'yellow')
def op_to_str(op:int, w=18) -> str: return colored(next((f"{op / d:{w-len(pr)-1}.2f} {pr}" for d,pr in [(1e12, "TOPs"), (1e9, "GOPs")] if op >= d), f"{op:{w-5}} OPs"), 'yellow')
def mem_to_str(mem:int, w=18) -> str: return colored(next((f"{mem / d:{w-len(pr)-1}.2f} {pr}" for d,pr in [(1e9, "GB"), (1e6, "MB")] if mem >= d), f"{mem:{w-7}} bytes"), 'yellow')
def op_tm_to_str(x:float, w=18) -> str: return colored(next((f"{x / d:{w-len(pr)-1}.2f} {pr}" for d,pr in [(1e12, "TOPs/sec"), (1e9, "GOPs/sec"), (1e6, "MOPs/sec")] if x >= d), f"{x:{w-9}.2f} OPs/sec"), 'blue')
def mem_tm_to_str(x:float, w=18) -> str: return colored(next((f"{x / d:{w-len(pr)-1}.2f} {pr}" for d,pr in [(1e9, "GB/sec"), (1e6, "MB/sec")] if x >= d), f"{x:{w-11}.2f} bytes/sec"), 'blue')
def op_mem_to_str(x:float, w=18) -> str: return colored(next((f"{x / d:{w-len(pr)-1}.2f} {pr}" for d,pr in [(1e9, "GOPs/GB"), (1e6, "MOPs/MB")] if x >= d), f"{x:{w-10}.2f} OPs/byte"), 'blue')

In [3]:
class Stats(contextlib.ContextDecorator):
  def __init__(self, prefix="", enabled=True, header=False): self.prefix, self.enabled, self.header = prefix, enabled, header
  def __enter__(self): self.tm, self.op, self.mem = time.perf_counter_ns(), GlobalCounters.global_ops, GlobalCounters.mem_used
  def __exit__(self, *exc):
    if not self.enabled: return
    if self.header: print(f"{'':40}{'Time':<18}  {'Compute':<18}  {'Memory':<18}  {'Compute Bandwidth':<18}  {'Memory Bandwidth':<18}  {'Arithmetic Intensity':<18}")
    tm, op, mem = (time.perf_counter_ns()-self.tm)*1e-9, GlobalCounters.global_ops-self.op, GlobalCounters.mem_used-self.mem
    op_tm, mem_tm, op_mem = op/(tm or 1e-20), mem/(tm or 1e-20), op/(mem or 1e-20)
    print(f"{self.prefix:<40}{time_to_str(tm)}, {op_to_str(op)}, {mem_to_str(mem)}, {op_tm_to_str(op_tm)}, {mem_tm_to_str(mem_tm)}, {op_mem_to_str(op_mem)}")

In [4]:
def apply_rope(x:Tensor, start_pos:int, base:float = 10000.0) -> Tensor:
  B, H, T, Dh = x.shape
  assert (Dh & 1) == 0, "RoPE requires an even head Dension"
  half = Dh // 2
  angles = (Tensor.arange(T, dtype="float32") + start_pos)[:, None] * (base ** (-(Tensor.arange(half, dtype="float32") / half)))[None, :]
  cos, sin = angles.cos().reshape(1, 1, T, half).cast(x.dtype), angles.sin().reshape(1, 1, T, half).cast(x.dtype)
  x_pairs = x.reshape(B, H, T, half, 2)
  return Tensor.stack(x_pairs[..., 0] * cos - x_pairs[..., 1] * sin,
                      x_pairs[..., 0] * sin + x_pairs[..., 1] * cos, dim=-1).reshape(B, H, T, Dh)

# Multi-Headed Attention (MHA)

In [5]:
# multi-headed attention
# https://arxiv.org/abs/1706.03762
class MHA:
  def __init__(self, dim:int, num_heads:int, max_context:int=0):
    self.num_heads = num_heads # H
    self.head_dim = dim // num_heads # Dh
    self.max_context = max_context # T

    self.attn_q = nn.Linear(dim, dim, bias=False) # (D,D)
    self.attn_k = nn.Linear(dim, dim, bias=False) # (D,D)
    self.attn_v = nn.Linear(dim, dim, bias=False) # (D,D)
    self.attn_o = nn.Linear(dim, dim, bias=False) # (D,D)

  def __call__(self, x:Tensor, start_pos:int=0, is_causal=False, kv_cache=False, rope=False) -> Tensor:
    # projections
    q, k, v = self.attn_q(x), self.attn_k(x), self.attn_v(x) # (B,T,D) -> (B,T,D)

    # reshape
    B, T, _ = x.shape
    q = q.reshape(B, T, self.num_heads, self.head_dim).transpose(1, 2)  # (B,T,D) -> (B,H,T,Dh)
    k = k.reshape(B, T, self.num_heads, self.head_dim).transpose(1, 2)  # (B,T,D) -> (B,H,T,Dh)
    v = v.reshape(B, T, self.num_heads, self.head_dim).transpose(1, 2)  # (B,T,D) -> (B,H,T,Dh)

    # positional embeddings
    if rope: q, k = apply_rope(q, start_pos), apply_rope(k, start_pos) # (B,H,T,Dh) -> (B,H,T,Dh)

    # kv cache
    if kv_cache and T > 1:
      if not hasattr(self, "kv_cache"):
        self.kv_cache = Tensor.zeros(2, self.max_context, B, self.num_heads, self.head_dim, dtype=k.dtype, device=k.device).contiguous().realize()
      self.kv_cache[:, start_pos:start_pos+T, :, :, :].assign(Tensor.stack(k, v))
      k = self.kv_cache[0, 0:start_pos+T, :, :, :]
      v = self.kv_cache[1, 0:start_pos+T, :, :, :]

    # compute attention
    qk = (q.matmul(k.transpose(-1, -2)) / math.sqrt(q.shape[-1])) # (B,H,T,Dh) (B,H,Dh,T) -> (B,H,T,T)
    if is_causal and T > 1:
      mask = Tensor.full((1, 1, T, start_pos+T), float("-inf"), dtype=x.dtype, device=x.device).triu(start_pos+1) # (B,H,T,T)
      qk = (qk + mask) # (B,H,T,T)
    s = qk.softmax(-1) # (B,H,T,T) -> (B,H,T,T)
    attn = s.matmul(v) # (B,H,T,T) (B,H,T,Dh) -> (B,H,T,Dh)
    attn = attn.transpose(1, 2).reshape(B, T, -1) # (B,H,T,Dh) -> (B,T,D)
    out = self.attn_o(attn) # (B,T,D) -> (B,T,D)
    return out

In [16]:
B, T, D, H = 1, 2, 4, 2
assert D % H == 0, f'hidden D {D=} must be divisible by number of heads {H=}'
D_h = D // H
x = Tensor.arange(B*T*D).reshape(B,T,D).cast(dtypes.float).realize()

In [7]:
with Stats("MHA Parameters:", header=True):
    model = MHA(D, H)
    for p in nn.state.get_parameters(model): p.realize()

with Stats("MHA Forward Pass:"):
    out = model(x).realize()

                                        Time                Compute             Memory              Compute Bandwidth   Memory Bandwidth    Arithmetic Intensity
MHA Parameters:                                  162.28 ms,         14939 OPs,         268 bytes,  92056.44 OPs/sec, 1651.46 bytes/sec,    55.74 OPs/byte
MHA Forward Pass:                                 28.44 ms,           356 OPs,          32 bytes,  12519.40 OPs/sec, 1125.34 bytes/sec,    11.12 OPs/byte


Why do the MHA parameters take up 268 bytes? Where does this number come from?
* The 4 model weights `self.attn_q`, `self.attn_k`, `self.attn_v`, `self.attn_o` are all matrices of shape $(D, D)$ with precision `float32`, i.e. $p=4$ bytes. This takes up $4 D^2 p = 256$ bytes.
* The 3 parameters `self.num_heads`, `self.head_dim`, `self.max_context` are all are integers with precision `int32`, i.e. `p=4` bytes. This takes up $3p = 12$ bytes.
* (256 bytes from model weights) + (12 bytes from the integer params) = 268 bytes, the exact number we got!

In [8]:
param_bytes = sum(x.numel() * x.dtype.itemsize for x in nn.state.get_parameters(model))
assert param_bytes == 256

Why does the MHA forward pass take 572 OPs?

To better understand it, we can add more fine grained stats to the model. We must carefully add realizes after each call to stat for the memory to actually be allocated and ops actually computed.

In [9]:
# multi-headed attention
# https://arxiv.org/abs/1706.03762
class MHA_Verbose:
  def __init__(self, dim:int, num_heads:int, max_context:int=0):

    self.num_heads = num_heads # H
    self.head_dim = dim // num_heads # Dh
    self.max_context = max_context # T

    self.attn_q = nn.Linear(dim, dim, bias=False) # (D,D)
    self.attn_k = nn.Linear(dim, dim, bias=False) # (D,D)
    self.attn_v = nn.Linear(dim, dim, bias=False) # (D,D)
    self.attn_o = nn.Linear(dim, dim, bias=False) # (D,D)

  def __call__(self, x:Tensor, start_pos:int=0, is_causal=False, kv_cache=False, rope=False) -> Tensor:
    # projections
    with Stats("q:"): q = self.attn_q(x).realize() # (B,T,D) -> (B,T,D)
    with Stats("k:"): k = self.attn_k(x).realize() # (B,T,D) -> (B,T,D)
    with Stats("v:"): v = self.attn_v(x).realize() # (B,T,D) -> (B,T,D)

    # reshape
    B, T, _ = x.shape
    q = q.reshape(B, T, self.num_heads, self.head_dim).transpose(1, 2)  # (B,T,D) -> (B,H,T,Dh)
    k = k.reshape(B, T, self.num_heads, self.head_dim).transpose(1, 2)  # (B,T,D) -> (B,H,T,Dh)
    v = v.reshape(B, T, self.num_heads, self.head_dim).transpose(1, 2)  # (B,T,D) -> (B,H,T,Dh)

    # positional embeddings
    if rope:
      with Stats("rope:"): q, k = apply_rope(q, start_pos).realize(), apply_rope(k, start_pos).realize() # (B,H,T,Dh) -> (B,H,T,Dh)

    # kv cache
    if kv_cache and T > 1:
      with Stats("kv cache:"):
        if not hasattr(self, "kv_cache"):
          self.kv_cache = Tensor.zeros(2, self.max_context, B, self.num_heads, self.head_dim, dtype=k.dtype, device=k.device).contiguous().realize()
        self.kv_cache[:, start_pos:start_pos+T, :, :, :].assign(Tensor.stack(k, v)).realize()
        k = self.kv_cache[0, 0:start_pos+T, :, :, :].realize()
        v = self.kv_cache[1, 0:start_pos+T, :, :, :].realize()

    # compute attention
    with Stats("qk:"): qk = (q.matmul(k.transpose(-1, -2)) / math.sqrt(q.shape[-1])).realize() # (B,H,T,Dh) (B,H,Dh,T) -> (B,H,T,T)
    if is_causal and T > 1:
      mask = Tensor.full((1, 1, T, start_pos+T), float("-inf"), dtype=x.dtype, device=x.device).triu(start_pos+1) # (B,H,T,T)
      qk = (qk + mask).realize() # (B,H,T,T)
    with Stats("softmax:"): s = qk.softmax(-1).realize() # (B,H,T,T) -> (B,H,T,T)
    with Stats("attn:"): attn = s.matmul(v).realize() # (B,H,T,T) (B,H,T,Dh) -> (B,H,T,Dh)
    attn = attn.transpose(1, 2).reshape(B, T, -1).realize() # (B,H,T,Dh) -> (B,T,D)
    with Stats("out:"): out = self.attn_o(attn).realize() # (B,T,D) -> (B,T,D)
    return out

In [10]:
print("MHA Parameters:")
with Stats("total:", header=True):
    model = MHA_Verbose(D, H)
    for p in nn.state.get_parameters(model): p.realize()

print("\nMHA Forward Pass:")
with Stats("total:"):
    out = model(x).realize()

MHA Parameters:
                                        Time                Compute             Memory              Compute Bandwidth   Memory Bandwidth    Arithmetic Intensity
total:                                            22.93 ms,         14940 OPs,           0 bytes, 651544.64 OPs/sec,    0.00 bytes/sec, 1494000000000000.00 GOPs/GB

MHA Forward Pass:
q:                                              673.04 us,            56 OPs,          32 bytes,  83204.44 OPs/sec, 47545.39 bytes/sec,     1.75 OPs/byte
k:                                              613.96 us,            56 OPs,          32 bytes,  91211.30 OPs/sec, 52120.74 bytes/sec,     1.75 OPs/byte
v:                                              511.25 us,            56 OPs,          32 bytes, 109535.45 OPs/sec, 62591.69 bytes/sec,     1.75 OPs/byte
qk:                                             963.38 us,            32 OPs,          32 bytes,  33216.56 OPs/sec, 33216.56 bytes/sec,     1.00 OPs/byte
softmax:                

* **Query projection**: Computing `q = self.attn_q(x)` where `x.shape = (B,T,D)` and `attn_q.weight.shape = (D,D)` produces `q.shape = (B,T,D)`. The matrix multiplication requires $BTD(2D-1) = 57$ OPs.

* **Key projection**: Computing `k = self.attn_k(x)` similarly requires $BTD(2D-1) = 56$ OPs.

* **Value projection**: Computing `v = self.attn_v(x)` similarly requires $BTD(2D-1) = 56$ OPs.

* **Attention scores**: Computing `qk = q.matmul(k.transpose(-1, -2)) / math.sqrt(D_h)` where `q.shape = (B,H,T,D_h)` and `k.transpose(-1, -2).shape = (B,H,D_h,T)` produces `qk.shape = (B,H,T,T)`. The matrix multiplication requires $BHT^2(2D_h-1)$ OPs and the element-wise division requires $BHT^2$ OPs, totaling $BHT^2(2D_h) = 32$ OPs.

* **Softmax**: Computing `s = qk.softmax(-1)` where `qk.shape = (B,H,T,T)` produces `s.shape = (B,H,T,T)`. For numerical stability, softmax is computed as `exp(qk - qk_max) / sum(exp(qk - qk_max))`. The tensor contains $BHT$ independent rows of length $T$:
    * **Max**: Finding `qk_max = max(qk)` per row: $T$ operations × $BHT$ rows = $BHT \cdot T$ OPs.
    * **Subtraction**: Computing `qk - qk_max` per element: $T$ operations × $BHT$ rows = $BHT \cdot T$ OPs.
    * **Exponentiation**: Computing `exp(qk - qk_max)`: $6T$ operations × $BHT$ rows = $BHT \cdot 6T$ OPs (each `exp` requires 6 OPs in tinygrad on Metal).
    * **Sum**: Summing $T$ values per row: $(T-1)$ operations × $BHT$ rows = $BHT \cdot (T-1)$ OPs.
    * **Division**: Dividing $T$ elements per row: $T$ operations × $BHT$ rows = $BHT \cdot T$ OPs.
    * **Total**: $BHT(10T - 1) = $ $76$ OPs.

* **Attention output**: Computing `attn = s.matmul(v)` where `s.shape = (B,H,T,T)` and `v.shape = (B,H,T,D_h)` produces `attn.shape = (B,H,T,D_h)`. This matrix multiplication requires $BHTD_h(2T-1) = $ $24$ OPs.

* **Output projection**: Computing `out = self.attn_o(attn)` where `attn.shape = (B,T,D)` and `attn_o.weight.shape = (D,D)` produces `out.shape = (B,T,D)`. This matrix multiplication requires $BTD(2D-1) = $ $56$ OPs.

# Group Query Attention

In [12]:
# group-query attention
# https://arxiv.org/abs/2305.13245
class GQA:
  def __init__(self, D:int, H:int, Hkv:int):
    self.num_heads = H # number of heads
    self.num_headskv = Hkv # number of kv heads
    self.head_dim = D // H # head dimension

    self.attn_q = nn.Linear(D, self.head_dim*H, bias=False) # (D,Dh*H)
    self.attn_k = nn.Linear(D, self.head_dim*Hkv, bias=False) # (D,Dh*Hkv)
    self.attn_v = nn.Linear(D, self.head_dim*Hkv, bias=False) # (D,Dh*Hkv)
    self.attn_o = nn.Linear(D, self.head_dim*H, bias=False) # (D,Dh*H)

  def __call__(self, x:Tensor, start_pos:int=0, is_causal=True) -> Tensor:
    # projections
    q = self.attn_q(x) # (B,T,D) -> (B,T,Dh*H)
    k = self.attn_k(x) # (B,T,D) -> (B,T,Dh*Hkv)
    v = self.attn_v(x) # (B,T,D) -> (B,T,Dh*Hkv)

    # reshape
    B, T, _ = x.shape
    q = q.reshape(B, T, self.num_heads, self.head_dim).transpose(1, 2)  # (B,T,D) -> (B,H,T,Dh)
    k = k.reshape(B, T, self.num_headskv, self.head_dim).transpose(1, 2)  # (B,T,Dh*Hkv) -> (B,Hkv,T,Dh)
    v = v.reshape(B, T, self.num_headskv, self.head_dim).transpose(1, 2)  # (B,T,Dh*Hkv) -> (B,Hkv,T,Dh)

    # gqa reshape
    k = k.repeat_interleave(self.num_heads // k.shape[-3], dim=-3) # (B,Hkv,T,Dh) -> (B,H,T,Dh)
    v = v.repeat_interleave(self.num_heads // v.shape[-3], dim=-3) # (B,Hkv,T,Dh) -> (B,H,T,Dh)

    # positional embeddings
    q, k = apply_rope(q, start_pos), apply_rope(k, start_pos) # (B,Hkv,T,Dh) -> (B,Hkv,T,Dh)

    # compute attention
    qk = q.matmul(k.transpose(-1, -2)) / math.sqrt(q.shape[-1]) # (B,H,T,Dh) (B,H,T,Dh) -> (B,H,T,T)
    if is_causal:
      mask = Tensor.full((1, 1, T, start_pos+T), float("-inf"), dtype=x.dtype, device=x.device).triu(start_pos+1) if T > 1 else None # (B,H,T,T)
      qk = qk + mask # (B,H,T,T)
    s = qk.softmax(-1) # (B,H,T,T) -> (B,H,T,T)
    attn = s.matmul(v) # (B,H,T,T) (B,H,T,Dh) -> (B,H,T,Dh)
    attn = attn.transpose(1, 2).reshape(B, T, -1) # (B,H,T,Dh) -> (B,T,D)
    out = self.attn_o(attn) # (B,T,D) -> (B,T,D)
    return out

In [13]:
B, T, D, H, Hkv = 2, 32, 256, 4, 2
assert D % H == 0, f'hidden D {D=} must be divisible by number of heads {H=}'
x = Tensor.arange(B*T*D).reshape(B,T,D).cast(dtypes.float).realize()

with Stats("GQA Weights:", header=True):
    model = GQA(D, H, Hkv)
    for p in nn.state.get_parameters(model): p.realize()

with Stats("GQA Forward Pass:"):
    out = model(x).realize()

                                        Time                Compute             Memory              Compute Bandwidth   Memory Bandwidth    Arithmetic Intensity
GQA Weights:                                     300.06 ms,      43819012 OPs,      786176 bytes,    146.04 MOPs/sec,        2.62 MB/sec,    55.74 OPs/byte
GQA Forward Pass:                                216.47 ms,      27657024 OPs,       65504 bytes,    127.77 MOPs/sec, 302607.64 bytes/sec,   422.22 OPs/byte


In [14]:
# multi-query attention
# https://arxiv.org/abs/1911.02150
class MQA(GQA):
  def __init__(self, D:int, H:int):
    # MQA is just like GQA but we set set n_kv_heads to 1
    super().__init__(D, H, 1)

In [15]:
B, T, D, H = 2, 32, 256, 4
assert D % H == 0, f'hidden D {D=} must be divisible by number of heads {H=}'

x = Tensor.arange(B*T*D).reshape(B,T,D).cast(dtypes.float).contiguous()
model = MQA(D, H)
out = model(x)
out.realize()

<Tensor <LB METAL (2, 32, 256) float ShapeTracker(views=(View(shape=(2, 32, 256), strides=(8192, 256, 1), offset=0, mask=None, contiguous=True),))> on METAL with grad None>

In [ ]:
B*H*T * ((c + 4)*T - 2)

In [28]:
76 / (B * H * T)

19.0

In [30]:
21/T-4

6.5